<a href="https://colab.research.google.com/github/G25ait2026/AIcopy/blob/main/codelfinal.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [19]:
import heapq
import itertools

class ManuscriptPuzzle:
    def __init__(self, start_state, goal_state, verbose=True):
        self.start_state = tuple(start_state)
        self.goal_state = tuple(goal_state)
        self.verbose = verbose

        self.goal_map = {val: (i // 3, i % 3) for i, val in enumerate(self.goal_state)}

        # Auto-fix if unsolvable
        if not self.is_solvable(self.start_state):
            print("Initial state UNSOLVABLE → Fixing parity...")
            self.start_state = self.make_solvable(self.start_state)

    # ---------- SOLVABILITY ----------
    def is_solvable(self, state):
        flat = [x for x in state if x != 'B']
        inv = 0
        for i in range(len(flat)):
            for j in range(i+1, len(flat)):
                if flat[i] > flat[j]:
                    inv += 1
        return inv % 2 == 0

    def make_solvable(self, state):
        state = list(state)
        idx = [i for i,v in enumerate(state) if v != 'B']
        state[idx[0]], state[idx[1]] = state[idx[1]], state[idx[0]]
        print("Converted Start State:", state)
        return tuple(state)

    # ---------- HEURISTIC ----------
    def h(self, node):
        dist = 0
        for i,val in enumerate(node):
            if val != 'B':
                r,c = i//3, i%3
                gr,gc = self.goal_map[val]
                dist += abs(r-gr) + abs(c-gc)
        return dist

    # ---------- PRINT BOARD ----------
    def print_board(self, state):
        for i in range(0,9,3):
            print(state[i:i+3])
        print()

    # ---------- NEIGHBORS ----------
    def get_neighbors(self, state):
        neighbors = []
        b = state.index('B')
        r,c = b//3, b%3

        for dr,dc in [(-1,0),(1,0),(0,-1),(0,1)]:
            nr,nc = r+dr, c+dc
            if 0<=nr<3 and 0<=nc<3:
                idx = nr*3+nc
                new_state = list(state)
                new_state[b], new_state[idx] = new_state[idx], new_state[b]
                neighbors.append(tuple(new_state))
        return neighbors

    # ---------- A* WITH TRACE ----------
    def a_star(self):
        counter = itertools.count()
        open_list = [(self.h(self.start_state), next(counter), self.start_state)]

        g_score = {self.start_state:0}
        parent = {self.start_state:None}

        step = 0

        while open_list:
            f,_,current = heapq.heappop(open_list)

            if self.verbose:
                print(f"--- Expansion Step {step} ---")
                print(f"g(n)={g_score[current]}  h(n)={self.h(current)}  f(n)={f}")
                self.print_board(current)

            if current == self.goal_state:
                print("Goal Reached!\n")
                return self.reconstruct_path(parent)

            for neighbor in self.get_neighbors(current):
                tentative_g = g_score[current] + 1

                if tentative_g < g_score.get(neighbor, float('inf')):
                    parent[neighbor] = current
                    g_score[neighbor] = tentative_g
                    f_val = tentative_g + self.h(neighbor)

                    heapq.heappush(open_list, (f_val, next(counter), neighbor))

                    if self.verbose:
                        print("Generated Neighbor:")
                        print(f"g={tentative_g}, h={self.h(neighbor)}, f={f_val}")
                        self.print_board(neighbor)

            step += 1

        return None

    # ---------- FINAL PATH ----------
    def reconstruct_path(self, parent):
        path = []
        cur = self.goal_state
        while cur:
            path.append(cur)
            cur = parent[cur]
        return path[::-1]


# ---------------- RUN ----------------
goal = [1,2,3,4,5,6,7,8,'B']
start = [1,2,3,'B',5,4,7,8,6]   # even unsolvable input works

solver = ManuscriptPuzzle(start, goal, verbose=True)
solution = solver.a_star()

print("========== FINAL SOLUTION PATH ==========\n")
for i,state in enumerate(solution):
    print(f"Move {i}:")
    solver.print_board(state)

print("Solved in", len(solution)-1, "moves.")


Initial state UNSOLVABLE → Fixing parity...
Converted Start State: [2, 1, 3, 'B', 5, 4, 7, 8, 6]
--- Expansion Step 0 ---
g(n)=0  h(n)=5  f(n)=5
(2, 1, 3)
('B', 5, 4)
(7, 8, 6)

Generated Neighbor:
g=1, h=6, f=7
('B', 1, 3)
(2, 5, 4)
(7, 8, 6)

Generated Neighbor:
g=1, h=6, f=7
(2, 1, 3)
(7, 5, 4)
('B', 8, 6)

Generated Neighbor:
g=1, h=6, f=7
(2, 1, 3)
(5, 'B', 4)
(7, 8, 6)

--- Expansion Step 1 ---
g(n)=1  h(n)=6  f(n)=7
('B', 1, 3)
(2, 5, 4)
(7, 8, 6)

Generated Neighbor:
g=2, h=5, f=7
(1, 'B', 3)
(2, 5, 4)
(7, 8, 6)

--- Expansion Step 2 ---
g(n)=1  h(n)=6  f(n)=7
(2, 1, 3)
(7, 5, 4)
('B', 8, 6)

Generated Neighbor:
g=2, h=7, f=9
(2, 1, 3)
(7, 5, 4)
(8, 'B', 6)

--- Expansion Step 3 ---
g(n)=1  h(n)=6  f(n)=7
(2, 1, 3)
(5, 'B', 4)
(7, 8, 6)

Generated Neighbor:
g=2, h=7, f=9
(2, 'B', 3)
(5, 1, 4)
(7, 8, 6)

Generated Neighbor:
g=2, h=7, f=9
(2, 1, 3)
(5, 8, 4)
(7, 'B', 6)

Generated Neighbor:
g=2, h=5, f=7
(2, 1, 3)
(5, 4, 'B')
(7, 8, 6)

--- Expansion Step 4 ---
g(n)=2  h(n)=5  f(